In [ ]:
# [1/5] Setup
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

!pip install -q scikit-learn joblib 2>/dev/null

import json
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

print('Ready.')

In [ ]:
# [2/5] CONFIG — select which model's scorer data to train on

MODEL = "Qwen8b"  # "Qwen4b" | "Qwen8b" | "Ministral8b"
SCORER_MODELS = [MODEL]

MODEL_OUTPUT = f"/content/drive/MyDrive/code/models/{MODEL}_scorer_lr.pkl"

FEATURE_NAMES = ["n_unique", "z3_le_count", "mean_bleu", "mean_bertscore", "backtrans_sim"]
LABEL_NAME = "z3_le_label"

In [ ]:
# [3/5] Load & combine data

X_all, y_all = [], []
n_loaded = 0

for model in SCORER_MODELS:
    path = f"/content/drive/MyDrive/code/data/results/{model}/k10/{model}_val_scorer.json"
    if not os.path.exists(path):
        print(f"  SKIP (not found): {path}")
        continue
    with open(path) as f:
        data = json.load(f)
    X = [[r[f] for f in FEATURE_NAMES] for r in data]
    y = [r[LABEL_NAME] for r in data]
    X_all.extend(X)
    y_all.extend(y)
    n_pos = sum(y)
    print(f"  {model}: {len(data)} rows, pos={n_pos} ({100*n_pos/len(data):.1f}%)")
    n_loaded += 1

if n_loaded == 0:
    raise RuntimeError("No scorer files found. Run scorer_data.ipynb first.")

X = np.array(X_all, dtype=np.float64)
y = np.array(y_all, dtype=np.int64)

print(f"\nTotal: {len(X)} rows  |  pos={int(sum(y))} ({100*sum(y)/len(y):.1f}%)")
print(f"Features: {FEATURE_NAMES}")

In [ ]:
# [4/5] 5-fold CV + final model

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---- CV evaluation ----
print("=" * 55)
print("  5-FOLD CROSS-VALIDATION")
print("=" * 55)
print()

fold_metrics = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    lr = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
    lr.fit(X_tr, y_tr)

    y_prob = lr.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "fold": fold + 1,
        "n_train": len(X_tr),
        "n_val": len(X_val),
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_prob),
    }
    fold_metrics.append(metrics)

    print(f"  Fold {fold+1}:  acc={metrics['accuracy']:.4f}  "
          f"prec={metrics['precision']:.4f}  rec={metrics['recall']:.4f}  "
          f"f1={metrics['f1']:.4f}  auc={metrics['roc_auc']:.4f}")

# ---- Summary ----
print()
print(f"  {'Metric':<15s} {'Mean':>8s}  {'Std':>8s}")
print("  " + "-" * 33)
for key in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    vals = [m[key] for m in fold_metrics]
    print(f"  {key:<15s} {np.mean(vals):>8.4f}  {np.std(vals):>8.4f}")

# ---- Train final model on all data ----
print()
print("  Training final model on all data ...")
final_lr = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
final_lr.fit(X, y)

print(f"  Coef: {dict(zip(FEATURE_NAMES, final_lr.coef_[0].round(4)))}")
print(f"  Intercept: {final_lr.intercept_[0]:.4f}")

In [ ]:
# [5/5] Save model

os.makedirs(os.path.dirname(MODEL_OUTPUT), exist_ok=True)
joblib.dump(final_lr, MODEL_OUTPUT)

print(f"Model saved to: {MODEL_OUTPUT}")
print()
print("To use in inference:")
print(f"  lr = joblib.load('{MODEL_OUTPUT}')")
print(f"  proba = lr.predict_proba([[n_unique, z3_le_count, mean_bleu, mean_bertscore, backtrans_sim]])[:, 1]")
print()

# Quick sanity check — predict first 5 samples
probas = final_lr.predict_proba(X[:5])[:, 1]
for i, p in enumerate(probas):
    print(f"  Sample {i}: proba={p:.4f}  label={y[i]}  features={X[i].round(4)}")